# Introduction to Vector Databases with ChromaDB

This notebook introduces the core ideas behind vector databases and walks through a practical example using ChromaDB to store text, generate embeddings, and run semantic similarity search.

## Learning goals
- Understand what a vector database is and why it is useful
- Learn how embeddings convert text into numeric vectors
- Create a persistent ChromaDB collection
- Add documents with metadata
- Run semantic search queries
- Understand how similarity scores work
- Explore practical use cases like RAG and recommendation systems

## What is a vector database?

A vector database stores data as high-dimensional vectors instead of only raw text. Each item is converted into an embedding, which captures its semantic meaning. Similar items end up close to each other in vector space.

This makes vector databases useful for:
- semantic search
- recommendation systems
- retrieval-augmented generation (RAG)
- clustering and similarity analysis

In this notebook, we will create a small ChromaDB collection, add documents, and query for similar text.


## Step 1: Install and import the required libraries

ChromaDB stores embeddings and supports fast similarity queries. For text embeddings, we often also use a sentence-transformer model.

If this is your first time running the notebook, uncomment the install line below.


In [ ]:
# Install the required libraries if they are not already available
# !pip install -q chromadb sentence-transformers

import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from pathlib import Path

print(f"Chroma version: {chromadb.__version__}")
print("Libraries imported successfully.")


## Step 2: Create a ChromaDB client and a persistent storage folder

There are two common ways to use ChromaDB:
- In-memory mode: good for quick experiments
- Persistent mode: saves your data on disk between runs

For a local demo, we use a folder named `./chroma_db` so the collection remains available even after the notebook stops running.


In [ ]:
import os
from pathlib import Path

persist_dir = Path("./chroma_db")
persist_dir.mkdir(exist_ok=True)

client = chromadb.Client(Settings(
    persist_directory=str(persist_dir),
    anonymized_telemetry=False,
))

print(f"Persistence folder created at: {persist_dir.resolve()}")
print("Client initialized successfully.")


## Step 3: Choose an embedding function

ChromaDB needs a way to convert text into vectors. The embedding function does this automatically.

In this example, we use Chroma's default embedding function. A real application may use a domain-specific or fine-tuned embedding model instead.

What is happening internally?
1. The text is passed to the model
2. The model converts it into a vector
3. ChromaDB stores the vector and indexes it for fast similarity lookup


In [ ]:
embedding_fn = embedding_functions.DefaultEmbeddingFunction()

sample_embedding = embedding_fn(["This is a sample sentence for embedding generation."])

print("Embedding function loaded.")
print(f"Embedding vector length: {len(sample_embedding[0])}")
print(f"First 5 values: {sample_embedding[0][:5]}")


## Step 4: Create or open a collection

A collection is a container for related documents. It stores:
- document text
- unique IDs
- metadata
- embeddings

We use the `get_or_create_collection` method so the notebook can be run multiple times without creating duplicates.


In [ ]:
collection = client.get_or_create_collection(
    name="demo_collection",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_fn,
)

print("Collection created or loaded: ", collection.name)
print("Collection count before add: ", collection.count())


## Step 5: Add sample documents

Each document should usually include:
- text content
- a unique ID
- optional metadata like topic, category, source, etc.

Why metadata matters:
- it helps with filtering
- it makes documents easier to organize
- it allows searches like "only AI-related documents"


In [ ]:
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A machine learning model learns patterns from data.",
    "Vector databases help retrieve semantically similar information.",
    "Cats are independent and curious animals.",
    "A neural network can process text and images.",
    "Deep learning models require large amounts of training data.",
    "Natural language processing powers conversational AI.",
    "Embeddings represent text as vectors in high-dimensional space.",
]

ids = [f"doc_{i}" for i in range(len(documents))]
metadatas = [
    {"topic": "general", "source": "demo"}
    for _ in documents
]

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
)

print("Inserted documents: ", collection.count())


## Step 6: Run a semantic search query

This is the core idea of vector search. We create a query text, convert it to an embedding, and ask ChromaDB to return the nearest neighbors based on similarity.

Important: we are not matching raw keywords only. Instead, the system compares meaning. So a query about "machine learning" can still match documents that use related ideas.


In [ ]:
query = "Find similar text about machine learning and retrieval systems."

results = collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print("Query: ", query)
print("\nTop matches:\n")

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"Document: {doc}")
    print(f"Metadata: {meta}")
    print(f"Distance: {distance:.4f}")
    print(f"Similarity score: {1 - distance:.4f}")
    print("-" * 60)


## Step 7: Understand the result

The query returns the documents with the smallest distance to the query vector. In a cosine space, a smaller distance usually means stronger semantic similarity.

Typical interpretation:
- distance close to 0 = very similar
- larger distance = less related

This is why vector databases can find similar text even when the exact words are different.


## Step 8: Run metadata-based filtering

ChromaDB also supports metadata filtering. This is useful when you only want to search a subset of the data, such as documents from a certain topic or source.


In [ ]:
# Add extra metadata for better filtering
collection.update(
    ids=["doc_0"],
    metadatas=[{"topic": "general", "source": "demo", "category": "animals"}],
)

collection.update(
    ids=["doc_1"],
    metadatas=[{"topic": "machine_learning", "source": "demo", "category": "ai"}],
)

collection.update(
    ids=["doc_2"],
    metadatas=[{"topic": "vector_db", "source": "demo", "category": "database"}],
)

query_ai = "Explain machine learning and neural networks"
results_ai = collection.query(
    query_texts=[query_ai],
    n_results=3,
    where={"category": "ai"},
    include=["documents", "metadatas", "distances"],
)

print("AI query:", query_ai)
print("Filtered results (category='ai'):")
for doc, meta, distance in zip(
    results_ai["documents"][0],
    results_ai["metadatas"][0],
    results_ai["distances"][0],
):
    print(f"- {doc} | metadata={meta} | distance={distance:.4f}")


## Step 9: Delete and update documents (optional but important)

In real projects, you often update old content or remove outdated records. ChromaDB supports both.


In [ ]:
# Update an existing document
collection.update(
    ids=["doc_0"],
    documents=["The quick brown fox jumps over the lazy dog — updated for demonstration."],
    metadatas=[{"topic": "general", "source": "demo", "category": "animals", "version": 2}],
)

# Add a temporary document and then delete it
collection.add(
    documents=["Temporary document for deletion example."],
    ids=["temp_doc"],
    metadatas=[{"topic": "temporary", "source": "demo"}],
)
print("Count before delete:", collection.count())

collection.delete(ids=["temp_doc"])
print("Count after delete:", collection.count())

updated_doc = collection.get(ids=["doc_0"], include=["documents", "metadatas"])
print("Updated document:", updated_doc["documents"][0])


## Step 10: Practical use cases

Vector databases are especially valuable in modern AI systems. Some examples: 

### 1. Semantic search
Search by meaning instead of exact keywords.

### 2. RAG (Retrieval-Augmented Generation)
An LLM retrieves the most relevant documents from a vector store and uses them as context before answering a user question.

### 3. Recommendations
Find similar products, articles, or users based on embeddings.

### 4. Duplicate detection
Detect near-duplicate records even when they are not word-for-word identical.

### 5. Clustering
Group similar documents into clusters for topic analysis.


## Step 11: Best practices

When building with vector databases, keep these recommendations in mind:

- clean your text before embedding
- chunk long documents into smaller, meaningful pieces
- use consistent metadata fields across your collection
- choose the right embedding model for your domain
- design queries carefully for better retrieval quality
- keep your embeddings and metadata in sync
- monitor retrieval quality with real examples


## Final takeaway

Vector databases store semantic representations of data and let us search by meaning rather than simple keyword matching. ChromaDB is a lightweight, beginner-friendly option for learning and prototyping vector search systems.

This small example is only the beginning. In real applications, vector stores are often used to power search systems, AI chatbots, enterprise knowledge bases, and recommendation services.

The key idea is simple: convert text into embeddings, store them in a vector database, and then retrieve semantically relevant results using similarity search.


In [ ]:
print("\n=== Final Summary ===")
print(f"Collection name: {collection.name}")
print(f"Collection count: {collection.count()}")
print("This notebook demonstrated vector storage, embedding generation, semantic search, and metadata filtering.")
